# ri-engine — Colab quickstart

**Pair with GitHub Codespaces** for the full CLI (plateau, runbook, terminal UI).

| Surface | Best for |
|---------|----------|
| **This notebook** | `improve()` API, quick experiments |
| **[Codespaces](https://github.com/codespaces)** | `ri-engine improve`, runbook, macro registry |

Docs: `docs/cloud_development.md` in the repo.

▶ **Runtime → Run all** (first run clones + installs; mock provider = no API key).

In [ ]:
# Config — change if you forked the repo
REPO_URL = "https://github.com/russfranky/recursive-intelligence.git"
REPO_BRANCH = "main"
PROJECT_DIR = "recursive-intelligence"  # clone target folder name

In [ ]:
# Clone and install (re-run after code updates)
import os
import subprocess
from pathlib import Path

if not Path(PROJECT_DIR).exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, PROJECT_DIR],
        check=True,
    )

%cd {PROJECT_DIR}
!pip install -q -e ".[all]"
print("✓ ri-engine installed")

In [ ]:
from ri_engine import improve, improve_template, improve_until_plateau, list_templates

print(f"{len(list_templates())} templates available:")
for t in list_templates():
    print(f"  · {t['id']}: {t['name']}")

## 1. State your desired outcome first

Edit the two cells below, then run them.

In [ ]:
SEED_PROMPT = """You are a helper. Answer questions."""

DESIRED_OUTCOME = """When this works, the AI will resolve customer billing issues in one conversation without escalation."""

In [ ]:
result = improve(
    seed_prompt=SEED_PROMPT,
    objective=DESIRED_OUTCOME,
    max_generations=3,
    population_size=4,
    provider="mock",  # change to "openai" if OPENAI_API_KEY is set
)

print(f"Fitness: {result.fitness:.1%}  ·  Generations: {result.generations}  ·  Converged: {result.converged}")
print("\n--- Improved prompt (copy to Codespaces / your AI tool) ---\n")
print(result.improved_prompt)

In [ ]:
# Template shortcut
tpl = improve_template("customer-support", max_generations=2, population_size=4)
print(f"Template fitness: {tpl.fitness:.1%}")
print(tpl.improved_prompt[:500], "…")

## 2. Optional — plateau cycling (slower)

Runs multiple improvement cycles until gains taper off.

In [ ]:
plateau = improve_until_plateau(
    SEED_PROMPT,
    DESIRED_OUTCOME,
    max_cycles=3,
    provider="mock",
)
print(f"Cycles: {plateau.cycles_run}  ·  Reason: {plateau.plateau_reason}  ·  Fitness: {plateau.final.fitness:.1%}")

## 3. Hand off to Codespaces (CLI + runbook)

Copy the improved prompt above, then in **GitHub Codespaces**:

```bash
cat > my_seed.txt << 'EOF'
<paste improved prompt>
EOF
ri-engine improve --seed my_seed.txt --goal "Your desired outcome" --until-plateau --runbook
```

Open Codespaces: GitHub repo → **Code** → **Codespaces** → **Create codespace**.

In [ ]:
# Export JSON for download / Codespaces handoff
import json
from pathlib import Path

payload = result.to_dict()
out = Path("output")
out.mkdir(exist_ok=True)
path = out / "colab_improved_prompt.json"
path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(f"Saved: {path.resolve()}")

try:
    from google.colab import files
    files.download(str(path))
except ImportError:
    print("(Download button available in Colab runtime)")

## Optional — OpenAI provider

Add `OPENAI_API_KEY` to Colab **Secrets**, then run:

In [ ]:
# Uncomment after setting Colab secret OPENAI_API_KEY
# from google.colab import userdata
# import os
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# result = improve(SEED_PROMPT, DESIRED_OUTCOME, provider="openai", max_generations=3)
# print(result.improved_prompt)